# Разность k порядка

## Задание
Загрузите файл в датафрейм _Pandas_ и назовите его _coderun_. Создайте в этом датафрейме столбец с датой и назовите его _date_.

Далее создайте новый датафрейм _df_, который получается в результате:


- группировки исходного датафрейма по дате

- расчету количества уникальных задач, решаемых в каждый день (столбец назовите problems)

- «превращение» столбца с датой из индекса в обычный столбец

А далее необходимо выполнить действия, типичные для работы, например, с временными рядами. 

Одно из важнейших понятий во временных рядах - стационарность. Для определения стационарности часто оценивают автокорреляции.
Если вдруг обнаруживается сильная автокорреляция, то стандартный способ борьбы - взятие разности k порядка. Проще говоря, из каждого значения ряда вычитают значение, взятое на k ячеек назад. После этого происходит повторная оценка автокорреляции.

Допустим, вы провели исследование и вас смущает автокорреляция 3 порядка. Вам необходимо:


- Создать столбец _prev_problems_ и записать в него значения столбца _problems_, сдвинутые на 3 назад. Если есть пропуски - туда вставляем 0.

- Создать столбец _diff_ и построчно посчитать разницу между _problems_ и _prev_problems_.

In [1]:
import pandas as pd

In [2]:
coderun = pd.read_csv('D:/GitHub_projects/Simulative_course/data/itresume-coderun.csv', encoding='1251')

In [3]:
coderun.head()

,id,created_at,problem_id,user_id,language_id
0,1,2021-04-07 06:06:20.000,13,10,3
1,2,2021-03-31 07:10:06.000,15,13,3
2,3,2021-04-04 14:55:26.000,1,6,3
3,4,2021-03-29 21:24:51.000,26,4,3
4,5,2021-03-30 11:29:12.000,21,18,3


In [4]:
coderun['date'] = pd.to_datetime(coderun['created_at']).dt.date
coderun

,id,created_at,problem_id,user_id,language_id,date
0,1,2021-04-07 06:06:20.000,13,10,3,2021-04-07
1,2,2021-03-31 07:10:06.000,15,13,3,2021-03-31
2,3,2021-04-04 14:55:26.000,1,6,3,2021-04-04
3,4,2021-03-29 21:24:51.000,26,4,3,2021-03-29
4,5,2021-03-30 11:29:12.000,21,18,3,2021-03-30
...,...,...,...,...,...,...
55717,55718,2022-05-17 09:00:51.743,101,171,2,2022-05-17
55718,55719,2022-05-17 09:01:04.210,101,171,2,2022-05-17
55719,55720,2022-05-17 09:02:06.203,101,171,2,2022-05-17
55720,55721,2022-05-17 09:03:45.265,102,171,2,2022-05-17


In [5]:
df = coderun.groupby('date'). \
    apply(lambda x: pd.Series(data = x['problem_id'].nunique(), index = ['problems'])). \
    reset_index()

C:\Users\79125\AppData\Local\Temp\ipykernel_19992\3513229609.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  apply(lambda x: pd.Series(data = x['problem_id'].nunique(), index = ['problems'])). \


In [6]:
df

,date,problems
0,2021-03-27,1
1,2021-03-28,1
2,2021-03-29,2
3,2021-03-30,1
4,2021-03-31,2
...,...,...
242,2022-05-13,41
243,2022-05-14,29
244,2022-05-15,32
245,2022-05-16,25


In [11]:
#Боремся с сильной автокорреляцией
df['prev_problems'] = df['problems'].shift(3).fillna(0).astype('Int64')
df['diff'] = (df['problems'] - df['prev_problems'])
df

,date,problems,prev_problems,diff
0,2021-03-27,1,0,1
1,2021-03-28,1,0,1
2,2021-03-29,2,0,2
3,2021-03-30,1,1,0
4,2021-03-31,2,1,1
...,...,...,...,...
242,2022-05-13,41,24,17
243,2022-05-14,29,56,-27
244,2022-05-15,32,51,-19
245,2022-05-16,25,41,-16
